# Topological Augmentation of Neural CDE Conditioning

This notebook tests whether persistent-homology summaries of the **frozen Phase 0 Neural CDE latent trajectory** provide useful predictive information beyond a simple non-topological trajectory-geometry control.

The Phase 0 Neural CDE is treated as the retained backbone. Phase 1 proceeds through three gates:

1. **Feasibility:** verify that H1 persistence is nontrivial, computationally practical, and not trivially identical to simple geometry.
2. **Controlled ablation:** compare a frozen-CDE condition against matched Geometry and TDA residual augmentations.
3. **Probabilistic validation:** compare validation diffusion loss, CRPS, Energy Score, and calibration under a common sampling seed.

Feature normalization is fit on the training period only. The held-out test split is not evaluated anywhere in this notebook.


## Setup


In [1]:
import os
import subprocess

REPO_PATH = "/kaggle/working/diffusion-portfolio"

REPO_URL = (
    "https://github.com/"
    "Amir-Entezari/"
    "diffusion-portfolio.git"
)

if not os.path.exists(
    os.path.join(
        REPO_PATH,
        ".git",
    )
):
    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            REPO_PATH,
        ],
        check=True,
    )

subprocess.run(
    [
        "git",
        "-C",
        REPO_PATH,
        "fetch",
        "origin",
        "main",
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        REPO_PATH,
        "reset",
        "--hard",
        "origin/main",
    ],
    check=True,
)

CURRENT_COMMIT = subprocess.check_output(
    [
        "git",
        "-C",
        REPO_PATH,
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

os.chdir(
    REPO_PATH
)

print(
    "Synced to latest main:"
)
print(
    CURRENT_COMMIT
)

subprocess.run(
    [
        "git",
        "log",
        "-1",
        "--oneline",
    ],
    check=True,
)


Cloning into '/kaggle/working/diffusion-portfolio'...


HEAD is now at 429dd70 Tighten Phase 1 experimental controls
Synced to latest main:
429dd707d06119e3b087213a48cf2e9f4f22886c
429dd70 Tighten Phase 1 experimental controls


From https://github.com/Amir-Entezari/diffusion-portfolio
 * branch            main       -> FETCH_HEAD


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

In [2]:
!pip install -q -e '.[research]'

import sys

SRC_PATH = f"{REPO_PATH}/src"

if SRC_PATH not in sys.path:
    sys.path.insert(
        0,
        SRC_PATH,
    )


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 24.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 3.3 MB/s eta 0:00:00
  Building editable for diffusion-portfolio (pyproject.toml) ... done


In [3]:
from pathlib import Path

import json
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from diffusion_portfolio.config import load_config
from diffusion_portfolio.data import collate_return_batch
from diffusion_portfolio.models.topology import (
    trajectory_geometry_features,
    trajectory_persistence_features,
)

from diffusion_portfolio.models.diffusion.build import build_diffusion
from diffusion_portfolio.data.preparation import prepare_datasets, load_standardizer


### Tests

Run the relevant regression tests before executing the Phase 1 experiments.


In [4]:
!pytest -q \
    tests/unit/test_cde.py \
    tests/unit/test_topology.py \
    tests/unit/test_precomputed_conditioning.py \
    tests/unit/test_conditional_diffusion_model.py \
    tests/unit/test_diffusion_training.py


........................................                                 [100%]


### Restore

Restore the Phase 0 CDE checkpoint and any previously backed-up Phase 1 outputs. Missing Phase 1 artifacts are simply recomputed.


In [5]:
import shutil
import kagglehub

HANDLE = "amirentezari/diffusion-portfolio"

RESTORES = {
    "phase0/neural_cde_rk4x4":
        "/kaggle/working/phase0_cde",

    "phase1/features":
        "/kaggle/working/phase1_features",

    "phase1/cde_control":
        "/kaggle/working/phase1_cde",

    "phase1/geometry_control":
        "/kaggle/working/phase1_geometry",

    "phase1/tda":
        "/kaggle/working/phase1_tda",
}

try:
    artifact_path = Path(
        kagglehub.dataset_download(
            HANDLE
        )
    )

    for relative, destination in (
        RESTORES.items()
    ):
        source = (
            artifact_path
            / relative
        )

        destination = Path(
            destination
        )

        if destination.exists():
            print(
                "Already available:",
                destination,
            )

        elif source.exists():
            shutil.copytree(
                source,
                destination,
            )

            print(
                "Restored:",
                destination,
            )

        else:
            print(
                "No backup:",
                relative,
            )

except Exception as error:
    print(
        "No backups restored.",
        error,
    )


Restored: /kaggle/working/phase0_cde
No backup: phase1/features
No backup: phase1/cde_control
No backup: phase1/geometry_control
No backup: phase1/tda


## 1. Feasibility gate

For each 60-day history, the frozen Neural CDE produces a latent trajectory

\[
H(t_1),\ldots,H(t_{60}) \in \mathbb{R}^{128}.
\]

Phase 1 treats these 60 latent states as a point cloud and computes Vietoris–Rips persistent homology through \(H_1\). Each homology dimension is summarized by persistence count, mean/std/max lifetime, persistence entropy, and the three largest lifetimes.

A seven-dimensional **simple geometry control** summarizes step lengths and distances from the trajectory centroid. This control is essential: TDA must add information beyond ordinary path scale and spread, not merely correlate with them.


In [6]:
CDE_DIR = Path(
    "/kaggle/working/phase0_cde"
)

if not CDE_DIR.exists():
    raise FileNotFoundError(
        "Phase 0 CDE artifact is required before Phase 1."
    )

cfg = load_config(
    CDE_DIR / "config.yaml"
)

checkpoint = torch.load(
    CDE_DIR / "best.pt",
    map_location="cpu",
    weights_only=False,
)

model = build_diffusion(
    cfg, n_assets=len(load_standardizer(CDE_DIR / "standardizer.npz").columns)
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

encoder = (
    model.history_encoder
    .to(device)
    .eval()
)

for parameter in encoder.parameters():
    parameter.requires_grad_(
        False
    )

datasets = prepare_datasets(
    cfg,
    standardizer_path=(
        CDE_DIR
        / "standardizer.npz"
    ),
)

print(
    "Device:",
    device,
)

print(
    "Checkpoint epoch:",
    checkpoint[
        "epoch"
    ],
)

print(
    "Validation loss:",
    checkpoint[
        "val_loss"
    ],
)


Device: cuda
Checkpoint epoch: 42
Validation loss: 1.445402071734143


In [7]:
loader = DataLoader(
    datasets.val,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_return_batch,
)

batch = next(
    iter(loader)
)

history = batch[
    "history"
].to(
    device
)

with torch.no_grad():
    trajectory = (
        encoder.encode_trajectory(
            history
        )
    )

if device.type == "cuda":
    torch.cuda.synchronize()

start = time.perf_counter()

tda_features = (
    trajectory_persistence_features(
        trajectory,
        max_homology_dim=1,
        top_k=3,
    )
)

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed = (
    time.perf_counter()
    - start
)

geometry_features = (
    trajectory_geometry_features(
        trajectory
    )
)

tda_np = (
    tda_features
    .detach()
    .cpu()
    .numpy()
)

geometry_np = (
    geometry_features
    .detach()
    .cpu()
    .numpy()
)

h1 = tda_np[
    :,
    8:16,
]

tda_std = tda_np.std(
    axis=0
)

nonconstant = (
    tda_std > 1e-12
)

nonconstant_indices = (
    np.flatnonzero(
        nonconstant
    )
)

correlations = np.corrcoef(
    np.column_stack(
        [
            tda_np[
                :,
                nonconstant
            ],
            geometry_np,
        ]
    ),
    rowvar=False,
)

n_tda = int(
    nonconstant.sum()
)

tda_geometry_corr = correlations[
    :n_tda,
    n_tda:,
]

h1_rows = np.flatnonzero(
    nonconstant_indices >= 8
)

max_corr = np.max(
    np.abs(
        tda_geometry_corr
    ),
    axis=1,
)

print(
    "Trajectory:",
    tuple(
        trajectory.shape
    ),
)

print(
    "Raw TDA features:",
    tuple(
        tda_features.shape
    ),
)

print(
    "Geometry features:",
    tuple(
        geometry_features.shape
    ),
)

print(
    "TDA time per sample:",
    elapsed
    / len(
        tda_features
    ),
)

print(
    "Samples with nonzero H1:",
    np.mean(
        np.any(
            np.abs(
                h1
            )
            > 1e-10,
            axis=1,
        )
    ),
)

print(
    "Constant raw TDA indices:",
    np.flatnonzero(
        ~nonconstant
    ).tolist(),
)

print(
    "Mean max |corr| with geometry:",
    float(
        max_corr.mean()
    ),
)

print(
    "H1 max |corr| with geometry:",
    max_corr[
        h1_rows
    ],
)


Trajectory: (32, 60, 128)
Raw TDA features: (32, 16)
Geometry features: (32, 7)
TDA time per sample: 0.012695778718750717
Samples with nonzero H1: 1.0
Constant raw TDA indices: [0]
Mean max |corr| with geometry: 0.5173689653617519
H1 max |corr| with geometry: [0.56539112 0.6333281  0.58756851 0.57514293 0.51452033 0.57514293
 0.27828132 0.32600599]


### Feasibility conclusion

The feasibility gate passes:

- extraction is cheap enough for one-time precomputation;
- \(H_1\) is nonzero across the probe batch and varies between samples;
- only the finite \(H_0\) bar-count feature is constant;
- TDA features show moderate, not near-perfect, correlation with the simple geometry controls.

This justifies a controlled predictive ablation, but it is **not** evidence that TDA improves forecasting.


## 2. Canonical feature precomputation

Precompute train and validation features once from the frozen seed-42 Phase 0 CDE.

The script uses the **saved Phase 0 standardizer**, obtains the canonical CDE condition through the same encoder forward path used in Phase 0, and uses the full latent trajectory only for Geometry/TDA extraction.

Geometry and TDA normalization statistics are estimated from the training split only. Constant training features are removed automatically. The test split is not materialized.


In [8]:
!PYTHONUNBUFFERED=1 python -u scripts/extract_features.py --kind topology \
    --cde-run-dir /kaggle/working/phase0_cde \
    --output-dir /kaggle/working/phase1_features \
    --batch-size 32 \
    --device cuda


Device: cuda
Checkpoint epoch: 42
Extracting train features...
Extracting validation features...

Train: 10516
Validation: 1256
CDE condition: 128
Geometry: 7
TDA: 15

Artifacts: /kaggle/working/phase1_features
TEST SPLIT WAS NOT TOUCHED.


In [9]:
train = np.load(
    "/kaggle/working/phase1_features/train.npz"
)

validation = np.load(
    "/kaggle/working/phase1_features/validation.npz"
)

with open(
    "/kaggle/working/phase1_features/metadata.json",
    encoding="utf-8",
) as handle:
    feature_metadata = json.load(
        handle
    )

print(
    "Train CDE:",
    train[
        "cde_condition"
    ].shape,
)

print(
    "Train geometry:",
    train[
        "geometry"
    ].shape,
)

print(
    "Train TDA:",
    train[
        "tda"
    ].shape,
)

print(
    "Validation CDE:",
    validation[
        "cde_condition"
    ].shape,
)

print(
    "Validation geometry:",
    validation[
        "geometry"
    ].shape,
)

print(
    "Validation TDA:",
    validation[
        "tda"
    ].shape,
)

print(
    "Geometry kept:",
    feature_metadata[
        "geometry_keep_indices"
    ],
)

print(
    "TDA kept:",
    feature_metadata[
        "tda_keep_indices"
    ],
)

print(
    "Train geometry max |mean|:",
    float(
        np.abs(
            train[
                "geometry"
            ].mean(
                axis=0
            )
        ).max()
    ),
)

print(
    "Train geometry max |std - 1|:",
    float(
        np.abs(
            train[
                "geometry"
            ].std(
                axis=0
            )
            - 1.0
        ).max()
    ),
)

print(
    "Train TDA max |mean|:",
    float(
        np.abs(
            train[
                "tda"
            ].mean(
                axis=0
            )
        ).max()
    ),
)

print(
    "Train TDA max |std - 1|:",
    float(
        np.abs(
            train[
                "tda"
            ].std(
                axis=0
            )
            - 1.0
        ).max()
    ),
)


Train CDE: (10516, 128)
Train geometry: (10516, 7)
Train TDA: (10516, 15)
Validation CDE: (1256, 128)
Validation geometry: (1256, 7)
Validation TDA: (1256, 15)
Geometry kept: [0, 1, 2, 3, 4, 5, 6]
TDA kept: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Train geometry max |mean|: 1.1608055849876564e-08
Train geometry max |std - 1|: 0.0
Train TDA max |mean|: 2.0314097071150172e-08
Train TDA max |std - 1|: 5.960464477539063e-08


## 3. Controlled frozen-CDE ablation

Train three otherwise matched diffusion models:

- **CDE:** cached 128-dimensional CDE condition only;
- **Geometry:** CDE condition plus a zero-initialized learned residual projection of 7 geometry features;
- **TDA:** CDE condition plus a zero-initialized learned residual projection of the retained TDA features.

The denoiser architecture and training protocol are identical. Variant-specific module construction is followed by an RNG reset inside the trainer so the three runs receive the same training corruption stream. The zero initialization ensures Geometry and TDA begin at the unaugmented CDE condition.


### 3.1 CDE control


In [10]:
!PYTHONUNBUFFERED=1 python -u scripts/train_feature_ablation.py \
    --variant cde \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase1_features \
    --output-dir /kaggle/working/phase1_cde \
    --device cuda \
    --resume


PHASE 1 FROZEN-CDE ABLATION
Variant: cde
Device: cuda
Input dimension: 128
Extra dimension: 0
Parameters: 826433
Epoch 001/050 | train=0.939267 | val=1.764985
Epoch 002/050 | train=0.823087 | val=1.733018
Epoch 003/050 | train=0.773741 | val=1.667340
Epoch 004/050 | train=0.767574 | val=1.668099
Epoch 005/050 | train=0.737292 | val=1.618344
Epoch 006/050 | train=0.724810 | val=1.612475
Epoch 007/050 | train=0.734012 | val=1.608986
Epoch 008/050 | train=0.710749 | val=1.600013
Epoch 009/050 | train=0.725750 | val=1.587595
Epoch 010/050 | train=0.727526 | val=1.602435
Epoch 011/050 | train=0.694413 | val=1.618143
Epoch 012/050 | train=0.685183 | val=1.609575
Epoch 013/050 | train=0.710996 | val=1.589171
Epoch 014/050 | train=0.683818 | val=1.607010
Epoch 015/050 | train=0.701027 | val=1.614698
Epoch 016/050 | train=0.697935 | val=1.627083
Epoch 017/050 | train=0.682714 | val=1.610811
Epoch 018/050 | train=0.676078 | val=1.614297
Epoch 019/050 | train=0.702400 | val=1.617279
Epoch 020/050

### 3.2 Geometry control


In [11]:
!PYTHONUNBUFFERED=1 python -u scripts/train_feature_ablation.py \
    --variant geometry \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase1_features \
    --output-dir /kaggle/working/phase1_geometry \
    --device cuda \
    --resume


PHASE 1 FROZEN-CDE ABLATION
Variant: geometry
Device: cuda
Input dimension: 135
Extra dimension: 7
Parameters: 827457
Epoch 001/050 | train=0.939234 | val=1.764775
Epoch 002/050 | train=0.823025 | val=1.732416
Epoch 003/050 | train=0.773627 | val=1.666840
Epoch 004/050 | train=0.767359 | val=1.666815
Epoch 005/050 | train=0.737029 | val=1.616800
Epoch 006/050 | train=0.724644 | val=1.611216
Epoch 007/050 | train=0.733647 | val=1.608112
Epoch 008/050 | train=0.710837 | val=1.598377
Epoch 009/050 | train=0.725492 | val=1.584806
Epoch 010/050 | train=0.727157 | val=1.602340
Epoch 011/050 | train=0.694593 | val=1.618681
Epoch 012/050 | train=0.685457 | val=1.607073
Epoch 013/050 | train=0.710599 | val=1.588721
Epoch 014/050 | train=0.683392 | val=1.616714
Epoch 015/050 | train=0.701065 | val=1.619094
Epoch 016/050 | train=0.697688 | val=1.631456
Epoch 017/050 | train=0.683266 | val=1.608771
Epoch 018/050 | train=0.675560 | val=1.614318
Epoch 019/050 | train=0.701059 | val=1.618064
Epoch 02

### 3.3 TDA augmentation


In [12]:
!PYTHONUNBUFFERED=1 python -u scripts/train_feature_ablation.py \
    --variant tda \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase1_features \
    --output-dir /kaggle/working/phase1_tda \
    --device cuda \
    --resume


PHASE 1 FROZEN-CDE ABLATION
Variant: tda
Device: cuda
Input dimension: 143
Extra dimension: 15
Parameters: 828481
Epoch 001/050 | train=0.939228 | val=1.764656
Epoch 002/050 | train=0.823017 | val=1.732222
Epoch 003/050 | train=0.773552 | val=1.666489
Epoch 004/050 | train=0.767289 | val=1.665929
Epoch 005/050 | train=0.736943 | val=1.616092
Epoch 006/050 | train=0.724601 | val=1.610808
Epoch 007/050 | train=0.733627 | val=1.607311
Epoch 008/050 | train=0.710587 | val=1.599293
Epoch 009/050 | train=0.725781 | val=1.586499
Epoch 010/050 | train=0.727061 | val=1.603681
Epoch 011/050 | train=0.694387 | val=1.621993
Epoch 012/050 | train=0.684650 | val=1.607655
Epoch 013/050 | train=0.710595 | val=1.592022
Epoch 014/050 | train=0.683632 | val=1.612578
Epoch 015/050 | train=0.701271 | val=1.619771
Epoch 016/050 | train=0.697740 | val=1.627184
Epoch 017/050 | train=0.682553 | val=1.613775
Epoch 018/050 | train=0.675694 | val=1.619690
Epoch 019/050 | train=0.701059 | val=1.606663
Epoch 020/05

### Training summary


In [13]:
training_runs = {
    "CDE": Path(
        "/kaggle/working/phase1_cde"
    ),
    "Geometry": Path(
        "/kaggle/working/phase1_geometry"
    ),
    "TDA": Path(
        "/kaggle/working/phase1_tda"
    ),
}

rows = []

for name, path in (
    training_runs.items()
):
    with (
        path / "summary.json"
    ).open(
        encoding="utf-8"
    ) as handle:
        summary = json.load(
            handle
        )

    projection_norm = np.nan

    if name != "CDE":
        checkpoint = torch.load(
            path / "best.pt",
            map_location="cpu",
            weights_only=False,
        )

        projection_norm = float(
            checkpoint[
                "model_state_dict"
            ][
                "residual_projection.weight"
            ].norm()
        )

    rows.append(
        {
            "Variant": name,
            "Best epoch": summary[
                "best_epoch"
            ],
            "Validation loss": summary[
                "best_val_loss"
            ],
            "Parameters": summary[
                "parameter_count"
            ],
            "Residual weight norm": (
                projection_norm
            ),
        }
    )

training_results = (
    pd.DataFrame(
        rows
    )
    .set_index(
        "Variant"
    )
)

display(
    training_results
)


,Best epoch,Validation loss,Parameters,Residual weight norm
Variant,,,,
CDE,9,1.587595,826433,NaN
Geometry,9,1.584806,827457,0.790539
TDA,9,1.586499,828481,0.920477


Canonical training-loss ranking:

\[
\text{Geometry} < \text{TDA} < \text{CDE}.
\]

Both residual projections move away from their zero initialization, confirming that the auxiliary branches are active rather than being ignored.


## 4. Validation-only probabilistic evaluation

Evaluate the best-validation checkpoint from each run with the same protocol used elsewhere in the project:

- 1,256 validation dates;
- 256 scenarios per date;
- sampling seed 12345;
- inverse transform to raw decimal excess returns;
- CRPS, Energy Score, and interval calibration.

The test split remains untouched.


### 4.1 CDE control


In [14]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --variant cde \
    --run-dir /kaggle/working/phase1_cde \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase1_features \
    --batch-size 32 \
    --device cuda


PHASE 1 VALIDATION EVALUATION
Variant: cde
Checkpoint epoch: 9
Checkpoint validation loss: 1.587595049363033
Validation samples: 1256
Scenarios per date: 256

cde: 100%|██████████████████████████████████████| 40/40 [02:21<00:00,  3.55s/it]

CRPS: 0.008562765237740667
Energy Score: 0.035273481422819386

Calibration:
{'level': 0.5, 'picp': 0.3391056263269639, 'ace': -0.1608943736730361}
{'level': 0.8, 'picp': 0.5904989384288747, 'ace': -0.20950106157112536}
{'level': 0.9, 'picp': 0.7047505307855626, 'ace': -0.19524946921443742}
{'level': 0.95, 'picp': 0.7816480891719745, 'ace': -0.16835191082802548}
{'level': 0.99, 'picp': 0.871881634819533, 'ace': -0.11811836518046703}

TEST SPLIT WAS NOT EVALUATED.


### 4.2 Geometry control


In [15]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --variant geometry \
    --run-dir /kaggle/working/phase1_geometry \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase1_features \
    --batch-size 32 \
    --device cuda


PHASE 1 VALIDATION EVALUATION
Variant: geometry
Checkpoint epoch: 9
Checkpoint validation loss: 1.5848057429501965
Validation samples: 1256
Scenarios per date: 256

geometry: 100%|█████████████████████████████████| 40/40 [02:21<00:00,  3.55s/it]

CRPS: 0.008539353885005624
Energy Score: 0.03519371409682867

Calibration:
{'level': 0.5, 'picp': 0.3391056263269639, 'ace': -0.1608943736730361}
{'level': 0.8, 'picp': 0.5938163481953291, 'ace': -0.2061836518046709}
{'level': 0.9, 'picp': 0.7078688959660298, 'ace': -0.19213110403397027}
{'level': 0.95, 'picp': 0.7838375796178344, 'ace': -0.16616242038216555}
{'level': 0.99, 'picp': 0.8742038216560509, 'ace': -0.11579617834394906}

TEST SPLIT WAS NOT EVALUATED.


### 4.3 TDA augmentation


In [16]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --variant tda \
    --run-dir /kaggle/working/phase1_tda \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase1_features \
    --batch-size 32 \
    --device cuda


PHASE 1 VALIDATION EVALUATION
Variant: tda
Checkpoint epoch: 9
Checkpoint validation loss: 1.5864986633039584
Validation samples: 1256
Scenarios per date: 256

tda: 100%|██████████████████████████████████████| 40/40 [02:22<00:00,  3.55s/it]

CRPS: 0.008562534999569103
Energy Score: 0.035281869093585626

Calibration:
{'level': 0.5, 'picp': 0.3416932059447983, 'ace': -0.1583067940552017}
{'level': 0.8, 'picp': 0.5936173036093418, 'ace': -0.2063826963906582}
{'level': 0.9, 'picp': 0.7081342887473461, 'ace': -0.19186571125265395}
{'level': 0.95, 'picp': 0.7841693205944799, 'ace': -0.1658306794055201}
{'level': 0.99, 'picp': 0.8746682590233545, 'ace': -0.11533174097664545}

TEST SPLIT WAS NOT EVALUATED.


### Final comparison


In [17]:
evaluation_rows = []

for name, path in (
    training_runs.items()
):
    with (
        path
        / "summary.json"
    ).open(
        encoding="utf-8"
    ) as handle:
        train_result = json.load(
            handle
        )

    with (
        path
        / "validation_probabilistic_metrics.json"
    ).open(
        encoding="utf-8"
    ) as handle:
        eval_result = json.load(
            handle
        )

    mean_abs_ace = float(
        np.mean(
            [
                abs(
                    item[
                        "ace"
                    ]
                )
                for item in eval_result[
                    "calibration"
                ]
            ]
        )
    )

    evaluation_rows.append(
        {
            "Variant": name,
            "Validation loss": train_result[
                "best_val_loss"
            ],
            "CRPS": eval_result[
                "crps_mean"
            ],
            "Energy Score": eval_result[
                "energy_score"
            ],
            "Mean |ACE|": (
                mean_abs_ace
            ),
        }
    )

phase1_results = (
    pd.DataFrame(
        evaluation_rows
    )
    .set_index(
        "Variant"
    )
)

display(
    phase1_results
)


,Validation loss,CRPS,Energy Score,Mean |ACE|
Variant,,,,
CDE,1.587595,0.008563,0.035273,0.170423
Geometry,1.584806,0.008539,0.035194,0.168234
TDA,1.586499,0.008563,0.035282,0.167544


## 5. Phase 1 decision

Canonical validation results from this complete fresh run:

| Variant | Validation loss ↓ | CRPS ↓ | Energy Score ↓ | Mean \(|ACE|\) ↓ |
|---|---:|---:|---:|---:|
| CDE | 1.587595 | 0.008562765 | 0.035273481 | 0.170423 |
| **CDE + Geometry** | **1.584806** | **0.008539354** | **0.035193714** | 0.168234 |
| CDE + TDA | 1.586499 | 0.008562535 | 0.035281869 | **0.167544** |

The simple Geometry control provides the strongest and most consistent improvement: it outperforms both CDE and TDA on validation diffusion loss, CRPS, and Energy Score.

TDA improves validation loss and calibration relative to the CDE-only control, but its CRPS is effectively unchanged and its Energy Score is slightly worse. More importantly, TDA does not outperform the much simpler Geometry control on the principal probabilistic forecasting metrics.

Therefore the evidence does not support attributing useful incremental predictive information to the persistent-homology representation.

**Decision:** reject the current persistent-homology augmentation and do not spend compute on multi-seed TDA tuning. The architecture carried forward remains the validated **Neural CDE-conditioned diffusion** from Phase 0.

The Geometry branch remains a useful scientific control rather than a retained architectural contribution.

The held-out test split remains untouched.

## Backup

Persist the canonical Phase 1 feature cache, controlled ablations, and validation-only evaluation artifacts to the central Kaggle dataset.


In [18]:
import shutil
import kagglehub

HANDLE = "amirentezari/diffusion-portfolio"

try:
    artifact_path = Path(
        kagglehub.dataset_download(
            HANDLE
        )
    )

except Exception as error:
    artifact_path = None

    print(
        "Existing backup unavailable:",
        error,
    )

staging = Path(
    "/kaggle/working/artifact_sync"
)

if staging.exists():
    shutil.rmtree(
        staging
    )

if artifact_path is not None:
    shutil.copytree(
        artifact_path,
        staging,
    )

else:
    staging.mkdir(
        parents=True,
        exist_ok=True,
    )

backups = {
    Path(
        "/kaggle/working/phase1_features"
    ): (
        staging
        / "phase1"
        / "features"
    ),

    Path(
        "/kaggle/working/phase1_cde"
    ): (
        staging
        / "phase1"
        / "cde_control"
    ),

    Path(
        "/kaggle/working/phase1_geometry"
    ): (
        staging
        / "phase1"
        / "geometry_control"
    ),

    Path(
        "/kaggle/working/phase1_tda"
    ): (
        staging
        / "phase1"
        / "tda"
    ),
}

for source, destination in (
    backups.items()
):
    if not source.exists():
        raise FileNotFoundError(
            source
        )

    if destination.exists():
        shutil.rmtree(
            destination
        )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copytree(
        source,
        destination,
    )

    print(
        f"Prepared: {source} -> {destination}"
    )

try:
    kagglehub.dataset_upload(
        HANDLE,
        str(
            staging
        ),
        version_notes=(
            "Complete Phase 1 TDA feasibility, "
            "controlled geometry ablation, and "
            "validation-only probabilistic evaluation"
        ),
    )

    print(
        "✅ Complete Phase 1 safely backed up"
    )

except Exception as error:
    print(
        "Backup upload skipped:",
        error,
    )


Prepared: /kaggle/working/phase1_features -> /kaggle/working/artifact_sync/phase1/features
Prepared: /kaggle/working/phase1_cde -> /kaggle/working/artifact_sync/phase1/cde_control
Prepared: /kaggle/working/phase1_geometry -> /kaggle/working/artifact_sync/phase1/geometry_control
Prepared: /kaggle/working/phase1_tda -> /kaggle/working/artifact_sync/phase1/tda
Uploading Dataset https://kaggle.com/datasets/amirentezari/diffusion-portfolio ...
More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmps84mt_e9/archive.zip


Uploading: 100%|██████████| 219M/219M [00:03<00:00, 67.1MB/s] 

Upload successful: /tmp/tmps84mt_e9/archive.zip (209MB)


Your dataset version has been created.
Files are being processed...
See at: https://kaggle.com/datasets/amirentezari/diffusion-portfolio
✅ Complete Phase 1 safely backed up
